In [6]:
import pandas as pd

# Load datasets
patient_df = pd.read_csv("/content/Patient_Data.csv")
billing_df = pd.read_csv("/content/Billing_Data.csv")

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Show dataset summary using info()

In [8]:
print("Patient Dataset Info:")
patient_df.info()

Patient Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


2. Select billing-related columns

In [9]:
billing_columns = ['PatientID', 'Department', 'Doctor', 'BillAmount']
billing_data = patient_df[billing_columns]

print("\nBilling Related Data:")
print(billing_data)


Billing Related Data:
   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN
5        101   Cardiology  Dr. Smith      5000.0


3. Drop administrative columns

In [10]:
patient_clean = patient_df.drop(
    columns=['ReceptionistID', 'CheckInTime']
)

print("\nAfter Dropping Administrative Columns:")
print(patient_clean)


After Dropping Administrative Columns:
   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN
5        101    Alice   Cardiology  Dr. Smith      5000.0


4. Total bill amount per department


In [11]:
department_bill = patient_df.groupby('Department')['BillAmount'].sum()

print("\nTotal Bill Amount Per Department:")
print(department_bill)


Total Bill Amount Per Department:
Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


5. Remove duplicate patient records

In [12]:
patient_no_duplicates = patient_df.drop_duplicates(
    subset='PatientID'
)

print("\nAfter Removing Duplicates:")
print(patient_no_duplicates)


After Removing Duplicates:
   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         NaN               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         NaN               2   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  


6. Fill missing BillAmount with mean

In [13]:
mean_bill = patient_df['BillAmount'].mean()

patient_df['BillAmount'] = patient_df['BillAmount'].fillna(mean_bill)

print("\nAfter Filling Missing BillAmount:")
print(patient_df)


After Filling Missing BillAmount:
   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John      5925.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose      5925.0               2   
5        101    Alice   Cardiology  Dr. Smith      5000.0               1   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  
5  2023-01-10 09:00  


 7. Merge billing dataset with patient dataset

In [14]:
merged_df = pd.merge(
    patient_df,
    billing_df,
    on='PatientID',
    how='left'
)

print("\nMerged Dataset:")
print(merged_df)


Merged Dataset:
   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John      5925.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose      5925.0               2   
5        101    Alice   Cardiology  Dr. Smith      5000.0               1   

        CheckInTime  InsuranceCovered  FinalAmount  
0  2023-01-10 09:00              2000         3000  
1  2023-01-11 10:30              1500         3500  
2  2023-01-12 11:00              2500         5000  
3  2023-01-13 12:00              3000         3200  
4  2023-01-14 08:45              1000         4000  
5  2023-01-10 09:00              2000         3000  


8. Concatenate new patients (row-wise)

In [15]:
new_patients = pd.DataFrame({
    'PatientID': [107, 108],
    'Name': ['John', 'Sara'],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Kumar', 'Dr. Rao'],
    'BillAmount': [4500, 5200],
    'ReceptionistID': [2, 1],
    'CheckInTime': ['2023-01-16 10:00', '2023-01-16 11:30']
})

all_patients = pd.concat(
    [patient_df, new_patients],
    ignore_index=True
)

print("\nAfter Row-wise Concatenation:")
print(all_patients)


After Row-wise Concatenation:
   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John      5925.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose      5925.0               2   
5        101    Alice   Cardiology  Dr. Smith      5000.0               1   
6        107     John   Cardiology  Dr. Kumar      4500.0               2   
7        108     Sara    Neurology    Dr. Rao      5200.0               1   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  
5  2023-01-10 09:00  
6  2023-01-16 10:00  
7  2023-01-16 11:30  


9. Concatenate billing category columns (column-wise)

In [16]:
billing_categories = billing_df[
    ['InsuranceCovered', 'FinalAmount']
]

column_concat = pd.concat(
    [patient_df, billing_categories],
    axis=1
)

print("\nAfter Column-wise Concatenation:")
print(column_concat)


After Column-wise Concatenation:
   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John      5925.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose      5925.0               2   
5        101    Alice   Cardiology  Dr. Smith      5000.0               1   

        CheckInTime  InsuranceCovered  FinalAmount  
0  2023-01-10 09:00            2000.0       3000.0  
1  2023-01-11 10:30            1500.0       3500.0  
2  2023-01-12 11:00            2500.0       5000.0  
3  2023-01-13 12:00            3000.0       3200.0  
4  2023-01-14 08:45            1000.0       4000.0  
5  2023-01-10 09:00               NaN          NaN  
